# Chapter 25 — Make the Problems Harder

**Companion to Applied AI**

Question: Why does an evaluation everyone passes tell you nothing about improving systems?

By the end of this notebook you will have:

- built an evaluation set with difficulty strata
- shown ceiling effects hiding real differences
- used discordant-pair analysis on the harder stratum

## What this notebook demonstrates
A synthetic stratified evaluation: easy tasks mask differences; harder tasks reveal them. Systems and scores are invented.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

seed: 42


## 1. Strata with known difficulty

In [2]:
strata = {"easy": (40, 0.95, 0.93), "medium": (30, 0.70, 0.62), "hard": (40, 0.45, 0.38)}
# (n_tasks, P_base_pass, P_challenger_pass)
results = {}
for name, (n, pb, pc) in strata.items():
    rb, rc = random.Random(SEED + hash(name)), random.Random(SEED + 100 + hash(name))
    b = [1 if rb.random() < pb else 0 for _ in range(n)]
    c = [1 if rc.random() < pc else 0 for _ in range(n)]
    results[name] = (b, c)
    print(f"{name:7s} base={sum(b)/n:.2f} challenger={sum(c)/n:.2f} gap={sum(c)/n - sum(b)/n:+.2f}")

easy    base=0.85 challenger=0.95 gap=+0.10
medium  base=0.73 challenger=0.60 gap=-0.13
hard    base=0.45 challenger=0.38 gap=-0.08


## 2. Ceiling effects: easy tasks declare a tie

In [3]:
b, c = results["easy"]
print(f"easy stratum: both ~{sum(b)/len(b):.2f}/{sum(c)/len(c):.2f} -> ranking impossible here")
assert abs(sum(b)/len(b) - sum(c)/len(c)) < 0.10

easy stratum: both ~0.85/0.95 -> ranking impossible here


## 3. Discordant pairs on hard tasks: the sign test

In [4]:
from math import comb
b, c = results["hard"]
wins = sum(1 for x, y in zip(b, c) if y > x)
losses = sum(1 for x, y in zip(b, c) if x > y)
n = wins + losses
p = sum(comb(n, i) for i in range(max(wins, losses), n + 1)) / 2**n if n else 1.0
print(f"hard stratum discordants: challenger {wins}, base {losses} (ties {len(b) - n}), two-sided p={p:.3f}")
print("Design lesson: spend evaluation budget where discordants live.")

hard stratum discordants: challenger 9, base 12 (ties 19), two-sided p=0.332
Design lesson: spend evaluation budget where discordants live.


## Interpretation
- Supports: evaluations everyone passes cannot rank improving systems; difficulty stratification plus discordant analysis shows where signal lives.
- Does NOT support: claims about any real benchmark.

## Try it yourself
1. Push easy accuracy to 1.00 for both and confirm zero discordants.
2. Reallocate: 10 easy + 70 hard, and compare CI width.
3. Split hard tasks by base-model pass rate and find where the challenger rescues.